
# Redes Neuronales desde Cero con NumPy  
## Desmitificando el aprendizaje para economistas

**Curso:** Inteligencia Artificial con Aplicaciones en Economía II  
**Enfoque:** programación desde cero, intuición económica y visualización pedagógica  
**Herramientas:** NumPy + Matplotlib  

---

## Objetivo del taller

En este notebook vamos a construir una red neuronal mínima **desde cero**, usando únicamente `NumPy` para la parte matemática.  

La idea no es competir con TensorFlow, PyTorch o Keras. La idea es entender **qué ocurre por dentro** cuando una red neuronal aprende.

---

## ¿Por qué un economista debería entender esto?

Muchas preguntas económicas pueden escribirse como problemas de decisión o predicción:

- ¿Un hogar caerá en pobreza?
- ¿Una empresa incumplirá un crédito?
- ¿Un estudiante desertará?
- ¿Una persona aceptará una política pública?
- ¿Un país entrará en estrés macroeconómico?

Un modelo clásico podría usar una regresión logística:

\[
P(y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1x_1 + \cdots + \beta_kx_k)}}
\]

Una neurona sigmoidal se parece mucho a eso:

\[
\hat{y} = \sigma(w_1x_1 + w_2x_2 + b)
\]

La diferencia es que las redes neuronales pueden apilar muchas de estas unidades para aprender relaciones más complejas.

---

## Conexión con Michael Nielsen

Michael Nielsen presenta las redes neuronales empezando por tres ideas muy poderosas:

1. el **perceptrón** como unidad decisora;
2. la **neurona sigmoidal** como versión suave y entrenable;
3. el aprendizaje como un proceso de **minimización de error** usando descenso del gradiente.

Eso es exactamente lo que haremos aquí, pero con ejemplos adaptados a economía.



---

# 1. Librerías

Usaremos:

- `numpy` para cálculos matemáticos;
- `matplotlib` para visualización;
- `pandas` solo para mostrar algunas tablas de resultados.

La red neuronal será implementada desde cero, sin `scikit-learn`, `tensorflow` ni `pytorch`.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)



---

# 2. Dataset sintético: riesgo económico simplificado

Vamos a simular un problema de decisión binaria inspirado en economía financiera.

Suponga que una institución quiere clasificar si una unidad económica tiene alto riesgo (`1`) o bajo riesgo (`0`) usando dos variables:

- \(x_1\): ingreso relativo normalizado;
- \(x_2\): nivel de deuda relativo normalizado.

Una intuición económica razonable es:

- mayor ingreso reduce el riesgo;
- mayor deuda aumenta el riesgo.

Para mantener el ejemplo visual, usaremos solo dos variables.


In [ ]:

def generate_economic_data(n=160, seed=42):
    """Genera un dataset sintético con dos variables económicas.

    Args:
        n: Número de observaciones.
        seed: Semilla para reproducibilidad.

    Returns:
        X: Matriz de predictores con columnas ingreso y deuda.
        y: Vector columna con clase binaria de riesgo.
    """
    rng = np.random.default_rng(seed)

    # Grupo 0: bajo riesgo, mayor ingreso y menor deuda
    low_risk = rng.normal(loc=[1.6, 0.8], scale=[0.45, 0.35], size=(n // 2, 2))

    # Grupo 1: alto riesgo, menor ingreso y mayor deuda
    high_risk = rng.normal(loc=[0.8, 1.7], scale=[0.40, 0.45], size=(n // 2, 2))

    # Unimos observaciones y etiquetas
    X = np.vstack([low_risk, high_risk])
    y = np.vstack([np.zeros((n // 2, 1)), np.ones((n // 2, 1))])

    # Mezclamos filas para no dejar los grupos ordenados
    idx = rng.permutation(n)
    return X[idx], y[idx]

X, y = generate_economic_data()

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)


In [ ]:

def plot_data(X, y, title="Datos sintéticos: riesgo económico"):
    """Grafica los puntos del dataset sintético."""
    plt.figure()
    plt.scatter(
        X[y.ravel() == 0, 0],
        X[y.ravel() == 0, 1],
        label="Bajo riesgo (0)",
        alpha=0.8
    )
    plt.scatter(
        X[y.ravel() == 1, 0],
        X[y.ravel() == 1, 1],
        label="Alto riesgo (1)",
        alpha=0.8
    )
    plt.xlabel("Ingreso relativo normalizado")
    plt.ylabel("Deuda relativa normalizada")
    plt.title(title)
    plt.legend()
    plt.show()

plot_data(X, y)



## Lectura económica

Visualmente, parece haber una frontera aproximada:

- arriba a la izquierda: menor ingreso y mayor deuda → mayor riesgo;
- abajo a la derecha: mayor ingreso y menor deuda → menor riesgo.

Un economista podría ver esto como un problema de elección binaria o clasificación.



---

# 3. La neurona desde cero: Perceptrón

## 3.1 Intuición

Un perceptrón calcula un índice lineal:

\[
z = w_1x_1 + w_2x_2 + b
\]

Luego aplica una regla de decisión dura:

\[
\hat{y} =
\begin{cases}
1, & z \geq 0\\
0, & z < 0
\end{cases}
\]

Esto se parece a una regla institucional con umbral:

> si el índice de riesgo supera cierto umbral, clasificamos como alto riesgo.

---

## Analogía económica

- \(w_1\): importancia relativa del ingreso;
- \(w_2\): importancia relativa de la deuda;
- \(b\): umbral o costo de entrada;
- salida: decisión binaria.


In [ ]:

def step_function(z):
    """Función escalón para el perceptrón."""
    return np.where(z >= 0, 1, 0)


def perceptron_predict(X, weights, bias):
    """Predice clases usando un perceptrón."""
    z = X @ weights + bias
    return step_function(z)


weights_manual = np.array([-1.0, 1.0])
bias_manual = 0.0

manual_pred = perceptron_predict(X, weights_manual, bias_manual)
manual_pred[:10].ravel()


In [ ]:

def plot_decision_boundary(X, y, weights, bias, title="Frontera de decisión"):
    """Grafica datos y frontera de decisión para dos variables."""
    plt.figure()
    plt.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], label="Bajo riesgo (0)", alpha=0.8)
    plt.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], label="Alto riesgo (1)", alpha=0.8)

    x1_values = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)

    if abs(weights[1]) > 1e-8:
        x2_values = -(weights[0] * x1_values + bias) / weights[1]
        plt.plot(x1_values, x2_values, label="Frontera", linewidth=2)

    plt.xlabel("Ingreso relativo normalizado")
    plt.ylabel("Deuda relativa normalizada")
    plt.title(title)
    plt.legend()
    plt.show()

plot_decision_boundary(
    X,
    y,
    weights_manual,
    bias_manual,
    title="Perceptrón con pesos manuales"
)



## Ejercicio rápido

Cambie manualmente los pesos y el sesgo:

- \(w_1\): peso del ingreso;
- \(w_2\): peso de la deuda;
- \(b\): umbral.

Observe cómo cambia la frontera.

**Pregunta:**  
¿Qué ocurre si el peso de la deuda es mucho mayor que el peso del ingreso?


In [ ]:

# EJERCICIO DEL ESTUDIANTE
# Cambie estos valores y vuelva a ejecutar la celda.

weights_try = np.array([-1.0, 1.5])
bias_try = -0.2

plot_decision_boundary(
    X,
    y,
    weights_try,
    bias_try,
    title="Frontera con pesos modificados por el estudiante"
)



---

# 4. De perceptrón a neurona sigmoidal

## Problema del perceptrón

El perceptrón cambia bruscamente de 0 a 1.  
Eso dificulta el aprendizaje gradual.

Si queremos ajustar pesos poco a poco mediante optimización, necesitamos una función suave.

## Función sigmoide

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

Esta función transforma cualquier valor real en un número entre 0 y 1.

En economía se parece a una probabilidad estimada:

\[
P(y=1|X)
\]

La suavidad permite calcular derivadas, y las derivadas permiten usar descenso del gradiente.


In [ ]:

def sigmoid(z):
    """Calcula la función sigmoide."""
    return 1 / (1 + np.exp(-z))


def sigmoid_derivative_from_output(a):
    """Calcula la derivada de la sigmoide usando su salida."""
    return a * (1 - a)


z_values = np.linspace(-8, 8, 300)

plt.figure()
plt.plot(z_values, sigmoid(z_values), label="Sigmoide")
plt.axhline(0.5, linestyle="--", alpha=0.7)
plt.axvline(0, linestyle="--", alpha=0.7)
plt.title("Función Sigmoide")
plt.xlabel("z")
plt.ylabel("σ(z)")
plt.legend()
plt.show()


In [ ]:

plt.figure()
plt.plot(z_values, step_function(z_values), label="Escalón / Perceptrón")
plt.plot(z_values, sigmoid(z_values), label="Sigmoide")
plt.title("Escalón vs Sigmoide")
plt.xlabel("z")
plt.ylabel("Salida")
plt.legend()
plt.show()



## Lectura económica

La sigmoide permite pasar de una regla rígida a una decisión probabilística.

En vez de decir:

> “esta empresa es riesgosa o no riesgosa”

podemos decir:

> “esta empresa tiene 0.78 de probabilidad estimada de alto riesgo”.

Ese cambio conecta redes neuronales con modelos de elección binaria.



---

# 5. Función de costo: medir qué tan mal predice el modelo

Para que una red aprenda, necesita una medida de error.

Usaremos el error cuadrático medio:

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
\]

Aunque en clasificación suele usarse entropía cruzada, el MSE es pedagógicamente útil para ver la lógica de minimización.

## Analogía económica

La función de costo es como una función de pérdida social:

- penaliza malas predicciones;
- guía la actualización de parámetros;
- define qué significa “mejorar”.


In [ ]:

def mse_cost(y_true, y_pred):
    """Calcula el error cuadrático medio."""
    return np.mean((y_true - y_pred) ** 2)


z_manual = X @ weights_manual.reshape(-1, 1) + bias_manual
prob_manual = sigmoid(z_manual)

initial_cost = mse_cost(y, prob_manual)

print("Costo inicial con pesos manuales:", initial_cost)



---

# 6. Visualización de la optimización: fronteras manuales

Antes de automatizar el aprendizaje, vamos a observar manualmente cómo cambia la frontera de decisión cuando cambian los pesos.

Esto ayuda a conectar:

- pesos;
- sesgo;
- frontera de decisión;
- y error del modelo.

En economía, esto equivale a cambiar parámetros de un modelo y observar cómo cambia la regla de clasificación.


In [ ]:

manual_configs = [
    {"weights": np.array([-0.5, 0.5]), "bias": 0.0, "name": "Frontera 1: pesos débiles"},
    {"weights": np.array([-1.0, 1.0]), "bias": 0.0, "name": "Frontera 2: pesos balanceados"},
    {"weights": np.array([-1.4, 1.8]), "bias": -0.3, "name": "Frontera 3: más peso a deuda"}
]

for config in manual_configs:
    plot_decision_boundary(
        X,
        y,
        config["weights"],
        config["bias"],
        title=config["name"]
    )



## Pregunta de discusión

¿Cuál de las tres fronteras parece separar mejor los grupos?

Desde una perspectiva económica:

- ¿debería pesar más el ingreso?
- ¿debería pesar más la deuda?
- ¿qué significa mover el sesgo?



---

# 7. Implementación de una red desde cero con Programación Orientada a Objetos

Ahora automatizaremos el aprendizaje.

Construiremos una clase `Network` que representa una neurona sigmoidal entrenable.

Esta red tendrá:

- pesos \(w\);
- sesgo \(b\);
- función sigmoide;
- función de costo;
- entrenamiento por descenso del gradiente.

---

## Regla de actualización

Queremos minimizar:

\[
C = \frac{1}{n}\sum(y - \hat{y})^2
\]

Usando descenso del gradiente:

\[
w \leftarrow w - \eta \frac{\partial C}{\partial w}
\]

\[
b \leftarrow b - \eta \frac{\partial C}{\partial b}
\]

donde \(\eta\) es la tasa de aprendizaje.

---

## Interpretación económica

La red ajusta sus parámetros de la misma forma en que un agente o institución aprende de errores pasados:

1. predice;
2. observa error;
3. ajusta regla de decisión;
4. repite.


In [ ]:

class Network:
    """Red neuronal mínima con una neurona sigmoidal.

    Esta clase implementa desde cero:
    - inicialización de pesos;
    - forward pass;
    - función de costo MSE;
    - entrenamiento por descenso del gradiente.

    Es equivalente a una regresión logística entrenada pedagógicamente con MSE.
    """

    def __init__(self, n_inputs, learning_rate=0.1, seed=42, zero_init=False):
        """Inicializa la red."""
        rng = np.random.default_rng(seed)

        # Si zero_init=True, se usan ceros; si no, valores pequeños aleatorios.
        if zero_init:
            self.weights = np.zeros((n_inputs, 1))
        else:
            self.weights = rng.normal(loc=0.0, scale=0.1, size=(n_inputs, 1))

        self.bias = 0.0
        self.learning_rate = learning_rate
        self.history = []
        self.weight_history = []

    def sigmoid(self, z):
        """Aplica la función sigmoide."""
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, a):
        """Calcula la derivada de la sigmoide usando la salida a."""
        return a * (1 - a)

    def forward(self, X):
        """Calcula la predicción de la red."""
        z = X @ self.weights + self.bias
        y_hat = self.sigmoid(z)
        return y_hat

    def cost(self, y_true, y_pred):
        """Calcula el error cuadrático medio."""
        return np.mean((y_true - y_pred) ** 2)

    def train(self, X, y, epochs=1000, store_every=50):
        """Entrena la red usando descenso del gradiente."""
        n = X.shape[0]

        for epoch in range(epochs):
            # Forward pass: predicción actual
            y_hat = self.forward(X)

            # Error de predicción
            error = y_hat - y

            # Derivada del MSE respecto a y_hat
            dcost_dyhat = 2 * error / n

            # Derivada de la sigmoide
            dyhat_dz = self.sigmoid_derivative(y_hat)

            # Regla de la cadena: derivada del costo respecto a z
            delta = dcost_dyhat * dyhat_dz

            # Gradiente respecto a los pesos
            dweights = X.T @ delta

            # Gradiente respecto al sesgo
            dbias = np.sum(delta)

            # Actualización de pesos y sesgo
            self.weights -= self.learning_rate * dweights
            self.bias -= self.learning_rate * dbias

            # Guardar historial para diagnóstico
            if epoch % store_every == 0 or epoch == epochs - 1:
                current_cost = self.cost(y, self.forward(X))
                self.history.append({"epoch": epoch, "cost": current_cost})
                self.weight_history.append((epoch, self.weights.copy(), self.bias))

    def predict_proba(self, X):
        """Devuelve probabilidades estimadas."""
        return self.forward(X)

    def predict(self, X, threshold=0.5):
        """Convierte probabilidades en clases 0/1."""
        return (self.predict_proba(X) >= threshold).astype(int)



---

# 8. Entrenamiento de la red

Entrenaremos la red y veremos cómo cae la función de costo.

En términos económicos, el modelo está aprendiendo una regla de clasificación que reduce errores de predicción.


In [ ]:

network = Network(n_inputs=2, learning_rate=0.5, seed=42)
network.train(X, y, epochs=2000, store_every=20)

history_df = pd.DataFrame(network.history)

display(history_df.head())
display(history_df.tail())


In [ ]:

plt.figure()
plt.plot(history_df["epoch"], history_df["cost"])
plt.title("Evolución de la función de costo")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.show()


In [ ]:

print("Pesos aprendidos:")
print(network.weights)

print("Sesgo aprendido:")
print(network.bias)

plot_decision_boundary(
    X,
    y,
    network.weights.ravel(),
    network.bias,
    title="Frontera de decisión aprendida por la red"
)



## Interpretación económica inicial

Los pesos aprendidos indican cómo el modelo pondera las variables:

- si el peso del ingreso es negativo, mayor ingreso reduce la probabilidad de alto riesgo;
- si el peso de la deuda es positivo, mayor deuda aumenta la probabilidad de alto riesgo;
- el sesgo ajusta el umbral general de clasificación.

Importante: esto es interpretación predictiva, no causal.



---

# 9. Cómo cambia la frontera durante el aprendizaje

Ahora observaremos algunos momentos del entrenamiento.

La red empieza con una frontera casi aleatoria y poco a poco ajusta sus pesos para separar mejor los grupos.


In [ ]:

selected_epochs = [0, 100, 300, 800, 1999]
history_dict = {epoch: (w, b) for epoch, w, b in network.weight_history}

for epoch in selected_epochs:
    available_epochs = np.array(list(history_dict.keys()))
    closest_epoch = available_epochs[np.argmin(np.abs(available_epochs - epoch))]
    w_epoch, b_epoch = history_dict[closest_epoch]

    plot_decision_boundary(
        X,
        y,
        w_epoch.ravel(),
        b_epoch,
        title=f"Frontera aprendida cerca de la época {closest_epoch}"
    )



## Pregunta de discusión

¿Qué observa en la evolución de la frontera?

- ¿El modelo empieza con una regla clara?
- ¿En qué momento parece estabilizarse?
- ¿La frontera final tiene sentido económico?



---

# 10. Evaluación simple

Como este es un problema de clasificación, revisaremos la proporción de aciertos.

No usaremos métricas avanzadas aquí porque el foco es entender la mecánica del aprendizaje.


In [ ]:

y_pred = network.predict(X)
accuracy = np.mean(y_pred == y)

print("Accuracy:", round(accuracy, 4))



---

# 11. What-if para economistas: ¿qué pasa si la tasa de aprendizaje es muy alta?

La tasa de aprendizaje \(\eta\) controla el tamaño de los pasos en la optimización.

## Analogía económica

Piense en una autoridad monetaria que ajusta la tasa de interés:

- si ajusta muy lento, tarda mucho en corregir;
- si ajusta demasiado agresivo, puede generar inestabilidad.

En redes neuronales ocurre algo parecido:

- \(\eta\) muy baja → aprendizaje lento;
- \(\eta\) razonable → convergencia estable;
- \(\eta\) muy alta → oscilaciones o inestabilidad.


In [ ]:

learning_rates = [0.01, 0.5, 5.0]
lr_histories = {}

for eta in learning_rates:
    temp_net = Network(n_inputs=2, learning_rate=eta, seed=42)
    temp_net.train(X, y, epochs=1000, store_every=10)
    lr_histories[eta] = pd.DataFrame(temp_net.history)

plt.figure()
for eta, hist in lr_histories.items():
    plt.plot(hist["epoch"], hist["cost"], label=f"eta={eta}")

plt.title("Efecto de la tasa de aprendizaje")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.legend()
plt.show()



## Preguntas

1. ¿Qué tasa de aprendizaje converge más rápido?
2. ¿Cuál parece más estable?
3. ¿Hay alguna que muestre comportamiento problemático?
4. ¿Cómo relacionaría esto con ajustes de política económica demasiado agresivos?



---

# 12. What-if: ¿qué pasa si inicializamos todos los pesos en cero?

En redes neuronales con muchas neuronas ocultas, inicializar todos los pesos en cero puede generar un problema de **simetría**:

- las neuronas empiezan iguales;
- reciben los mismos gradientes;
- aprenden lo mismo;
- se pierde diversidad de representación.

## Matiz importante

En este ejemplo tenemos una sola neurona sigmoidal.  
En una sola neurona, inicializar en cero no impide necesariamente aprender.  
El problema fuerte aparece cuando hay múltiples neuronas ocultas que deberían aprender patrones distintos.

Este matiz es importante para no repetir reglas de memoria sin entenderlas.


In [ ]:

zero_net = Network(n_inputs=2, learning_rate=0.5, seed=42, zero_init=True)
zero_net.train(X, y, epochs=1000, store_every=20)

zero_history = pd.DataFrame(zero_net.history)

plt.figure()
plt.plot(zero_history["epoch"], zero_history["cost"])
plt.title("Entrenamiento con pesos iniciales en cero")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.show()

print("Pesos aprendidos desde cero:")
print(zero_net.weights)
print("Sesgo aprendido desde cero:")
print(zero_net.bias)



## Pregunta de reflexión

¿Por qué esta red de una sola neurona sí logra aprender con pesos en cero, pero una red con varias neuronas ocultas podría tener problemas?

Escriba la respuesta con sus palabras.



---

# 13. Interpretación económica de los resultados

Vamos a interpretar los parámetros aprendidos como si fueran señales similares a impactos marginales o elasticidades, pero con una advertencia importante:

> Los pesos de una red neuronal son parámetros predictivos, no efectos causales.

En este ejemplo, como tenemos una sola neurona sigmoidal, la interpretación es cercana a la de una regresión logística:

\[
\hat{y} = \sigma(w_1 x_1 + w_2 x_2 + b)
\]

donde:

- \(w_1\) indica cómo cambia el índice latente cuando aumenta el ingreso;
- \(w_2\) indica cómo cambia el índice latente cuando aumenta la deuda;
- \(b\) define el umbral base del modelo.


In [ ]:

param_table = pd.DataFrame({
    "variable": ["Ingreso relativo", "Deuda relativa"],
    "peso_aprendido": network.weights.ravel()
})

display(param_table)

print("Sesgo aprendido:", network.bias)



## Lectura económica esperada

Si el modelo aprendió correctamente:

- el peso del ingreso debería ser negativo;
- el peso de la deuda debería ser positivo.

Eso sería coherente con la intuición:

- más ingreso → menor riesgo;
- más deuda → mayor riesgo.

## Advertencia causal

No podemos decir que aumentar ingreso “causa” menor riesgo solo por este peso.  
La red está aprendiendo una relación predictiva dentro de un dataset sintético.

En economía aplicada real, la interpretación causal requiere diseño de identificación.



---

# 14. Ejercicios finales

## Ejercicio 1: cambie la tasa de aprendizaje

Entrene tres redes con:

- \(\eta = 0.001\)
- \(\eta = 0.1\)
- \(\eta = 10\)

Responda:

1. ¿Cuál aprende más lento?
2. ¿Cuál parece más estable?
3. ¿Cuál se comporta peor?

---

## Ejercicio 2: cambie el dataset

Modifique la función `generate_economic_data()` para que los grupos se traslapen más.

Responda:

1. ¿El modelo sigue separando bien?
2. ¿Qué pasa con el accuracy?
3. ¿Qué significa esto en términos económicos?

---

## Ejercicio 3: cambie el umbral de clasificación

Actualmente usamos umbral 0.5.

Pruebe:

- 0.3
- 0.7

Responda:

1. ¿Qué pasa con los falsos positivos?
2. ¿Qué pasa con los falsos negativos?
3. ¿Qué umbral usaría una entidad financiera conservadora?

---

## Ejercicio 4: conecte con economía

Explique en máximo 8 líneas cómo se relacionan:

- pesos;
- sesgo;
- función sigmoide;
- descenso del gradiente;

con un modelo de elección binaria en economía.


In [ ]:

# ESPACIO PARA LOS EJERCICIOS DEL ESTUDIANTE

# Puede reutilizar la clase Network para experimentar.



---

# 15. Cierre conceptual

En este notebook construimos una red neuronal mínima desde cero.

## Ideas clave

1. Un perceptrón es una regla de decisión binaria con umbral.
2. La sigmoide suaviza esa decisión y permite derivadas.
3. El aprendizaje consiste en minimizar una función de costo.
4. El descenso del gradiente actualiza pesos en dirección de menor error.
5. La tasa de aprendizaje controla la estabilidad del proceso.
6. Los pesos pueden leerse como señales de importancia predictiva.
7. La interpretación causal requiere mucho más que una red entrenada.

## Mensaje final

Una red neuronal no es magia.  
Es una máquina de optimización que ajusta parámetros para reducir errores.

Para un economista, lo importante no es solo programarla, sino entender:

- qué decisión representa;
- qué información usa;
- qué error minimiza;
- qué supuestos implícitos tiene;
- y si su comportamiento tiene sentido económico.
